# Análise Exploratória

Nesta etapa será realizada a análise exploratória do dataset integrado, buscando identificar padrões, diferenças, tendências e possíveis anomalias relacionadas à produtividade agrícola.

As análises são orientadas pelas perguntas definidas no projeto e abrangem:

* produtividade por cultura;
* produtividade por país;
* evolução temporal;
* associação entre produtividade e temperatura;
* associação entre produtividade e precipitação;
* associação entre produtividade e uso de pesticidas;
* identificação de valores extremos e possíveis anomalias.

Os resultados desta etapa serão utilizados como base para a construção dos indicadores consolidados no notebook seguinte e, posteriormente, para o desenvolvimento do dashboard no Power BI.

> **Observação:** as associações observadas entre as variáveis não são interpretadas como relações de causalidade.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

yield_integrado = pd.read_csv('../dados/yield_integrado.csv')

yield_integrado.info()

## 1. Diagnóstico analítico do dataset integrado

In [ ]:
print(f"Registros: {len(yield_integrado):,}")
print(f"Colunas: {yield_integrado.shape[1]}")
print(f"Período: {yield_integrado['Year'].min()}–{yield_integrado['Year'].max()}")
print(f"Países/áreas: {yield_integrado['Area'].nunique()}")
print(f"Culturas: {yield_integrado['Item'].nunique()}")

In [ ]:
yield_integrado['Item'].value_counts().sort_index()

### 1.1 Cobertura das variáveis auxiliares

In [ ]:
cobertura = pd.DataFrame({
    'registros': len(yield_integrado),
    'disponiveis': [
        yield_integrado['rainfall_mm'].notna().sum(),
        yield_integrado['pesticides_tonnes'].notna().sum(),
        yield_integrado['avg_temp'].notna().sum()
    ],
    'ausentes': [
        yield_integrado['rainfall_mm'].isna().sum(),
        yield_integrado['pesticides_tonnes'].isna().sum(),
        yield_integrado['avg_temp'].isna().sum()
    ]
}, index=[
    'rainfall_mm',
    'pesticides_tonnes',
    'avg_temp'
])

cobertura['cobertura_%'] = (
    cobertura['disponiveis'] /
    cobertura['registros'] * 100
).round(2)

cobertura

## 2. Produtividade por cultura

A produtividade é analisada inicialmente por cultura, considerando medidas de tendência central e dispersão. A mediana será utilizada como uma referência importante devido à presença de valores extremos na base.

In [ ]:
produtividade_cultura = (
    yield_integrado
    .groupby('Item')['Value']
    .agg(
        registros='count',
        media='mean',
        mediana='median',
        minimo='min',
        maximo='max'
    )
    .sort_values('mediana', ascending=False)
)

produtividade_cultura

### 2.1 Distribuição da produtividade

In [ ]:
plt.figure(figsize=(12, 7))

yield_integrado.boxplot(
    column='Value',
    by='Item',
    vert=False
)

plt.title('Distribuição da produtividade por cultura')
plt.suptitle('')
plt.xlabel('Produtividade (hg/ha)')
plt.ylabel('Cultura')
plt.show()

In [ ]:
limite_99 = yield_integrado['Value'].quantile(0.99)

dados_boxplot = yield_integrado[
    yield_integrado['Value'] <= limite_99
]

plt.figure(figsize=(12, 7))

dados_boxplot.boxplot(
    column='Value',
    by='Item',
    vert=False
)

plt.title('Distribuição da produtividade por cultura — até o percentil 99')
plt.suptitle('')
plt.xlabel('Produtividade (hg/ha)')
plt.ylabel('Cultura')
plt.show()

### 2.2 Identificação de outliers

In [ ]:
quartis_cultura = (
    yield_integrado
    .groupby('Item')['Value']
    .quantile([0.25, 0.50, 0.75])
    .unstack()
    .rename(columns={
        0.25: 'Q1',
        0.50: 'Mediana',
        0.75: 'Q3'
    })
)

quartis_cultura['IQR'] = (
    quartis_cultura['Q3'] -
    quartis_cultura['Q1']
)

quartis_cultura['limite_inferior'] = (
    quartis_cultura['Q1'] -
    1.5 * quartis_cultura['IQR']
)

quartis_cultura['limite_superior'] = (
    quartis_cultura['Q3'] +
    1.5 * quartis_cultura['IQR']
)

quartis_cultura

In [ ]:
outliers_cultura = []

for cultura, grupo in yield_integrado.groupby('Item'):
    limite_inferior = quartis_cultura.loc[cultura, 'limite_inferior']
    limite_superior = quartis_cultura.loc[cultura, 'limite_superior']

    outliers = grupo[
        (grupo['Value'] < limite_inferior) |
        (grupo['Value'] > limite_superior)
    ]

    outliers_cultura.append({
        'Item': cultura,
        'outliers': len(outliers),
        'total_registros': len(grupo),
        'percentual_outliers': len(outliers) / len(grupo) * 100
    })

outliers_cultura = (
    pd.DataFrame(outliers_cultura)
    .sort_values('outliers', ascending=False)
)

outliers_cultura

## 3. Produtividade por país

A produtividade é analisada por combinação entre cultura e país. Para reduzir o efeito de grupos com poucas observações, são considerados apenas os grupos com pelo menos 20 registros.

A mediana é utilizada como principal referência para comparação entre os países.

In [ ]:
pais_cultura = (
    yield_integrado
    .groupby(['Item', 'Area'])['Value']
    .agg(
        registros='count',
        media='mean',
        mediana='median',
        minimo='min',
        maximo='max'
    )
    .reset_index()
)

pais_cultura_filtrado = pais_cultura[
    pais_cultura['registros'] >= 20
]

pais_cultura_filtrado.head()

In [ ]:
cobertura_culturas = (
    pais_cultura_filtrado
    .groupby('Item')['Area']
    .nunique()
    .sort_values(ascending=False)
)

cobertura_culturas

In [ ]:
maiores_por_cultura = (
    pais_cultura_filtrado
    .sort_values(['Item', 'mediana'], ascending=[True, False])
    .groupby('Item')
    .head(3)
)

menores_por_cultura = (
    pais_cultura_filtrado
    .sort_values(['Item', 'mediana'], ascending=[True, True])
    .groupby('Item')
    .head(3)
)

maiores_por_cultura

In [ ]:
menores_por_cultura

## 4. Evolução temporal

A evolução da produtividade é analisada ao longo do período de 1961 a 2016, utilizando a mediana anual por cultura como principal referência.

A análise busca identificar tendências de crescimento ou redução da produtividade ao longo do período. A quantidade de países representados em cada ano também é considerada, pois a cobertura da base varia entre culturas e ao longo do tempo.

In [ ]:
produtividade_ano = (
    yield_integrado
    .groupby(['Year', 'Item'])['Value']
    .median()
    .reset_index(name='produtividade_mediana')
)

produtividade_ano.head(20)

In [ ]:
produtividade_ano_pivot = (
    produtividade_ano
    .pivot(
        index='Year',
        columns='Item',
        values='produtividade_mediana'
    )
)

produtividade_ano_pivot

### 4.1 Evolução anual por cultura

In [ ]:
plt.figure(figsize=(14, 8))

for cultura in produtividade_ano_pivot.columns:
    plt.plot(
        produtividade_ano_pivot.index,
        produtividade_ano_pivot[cultura],
        label=cultura
    )

plt.title('Evolução da produtividade mediana por cultura (1961–2016)')
plt.xlabel('Ano')
plt.ylabel('Produtividade (hg/ha)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4.2 Comparação entre o início e o fim do período

A comparação entre 1961 e 2016 permite observar a variação da produtividade mediana de cada cultura ao longo do período analisado.

In [ ]:
analise_temporal = (
    produtividade_ano_pivot.loc[[1961, 2016]]
    .T
    .rename(columns={
        1961: 'produtividade_1961',
        2016: 'produtividade_2016'
    })
)

analise_temporal['variacao_absoluta'] = (
    analise_temporal['produtividade_2016']
    - analise_temporal['produtividade_1961']
)

analise_temporal['crescimento_percentual'] = (
    (
        analise_temporal['produtividade_2016']
        / analise_temporal['produtividade_1961']
        - 1
    ) * 100
).round(2)

analise_temporal.sort_values(
    'crescimento_percentual',
    ascending=False
)

In [ ]:
cobertura_temporal = (
    yield_integrado
    .groupby(['Year', 'Item'])['Area']
    .nunique()
    .reset_index(name='paises')
)

cobertura_temporal_pivot = (
    cobertura_temporal
    .pivot(
        index='Year',
        columns='Item',
        values='paises'
    )
)

cobertura_temporal_pivot

In [ ]:
cobertura_resumo = (
    cobertura_temporal
    .groupby('Item')['paises']
    .agg(
        minimo='min',
        mediana='median',
        maximo='max'
    )
    .sort_values('mediana', ascending=False)
)

cobertura_resumo

> **Limitação da análise temporal:** a quantidade de países representados varia ao longo dos anos e entre culturas. Para algumas culturas, a cobertura permanece relativamente estável, enquanto outras apresentam expansão significativa, principalmente a partir da década de 1990. Portanto, as medianas anuais representam a produtividade dos países disponíveis em cada ano e não necessariamente a evolução de um painel fixo de países.

### 4.3 Extremos da evolução temporal

Para cada cultura, são identificados os anos em que a produtividade mediana anual apresentou seus menores e maiores valores no período analisado.

In [ ]:
extremos_temporais = []

for cultura in produtividade_ano['Item'].unique():
    dados = produtividade_ano[produtividade_ano['Item'] == cultura]

    idx_min = dados['produtividade_mediana'].idxmin()
    idx_max = dados['produtividade_mediana'].idxmax()

    extremos_temporais.append({
        'Item': cultura,
        'ano_minimo': dados.loc[idx_min, 'Year'],
        'valor_minimo': dados.loc[idx_min, 'produtividade_mediana'],
        'ano_maximo': dados.loc[idx_max, 'Year'],
        'valor_maximo': dados.loc[idx_max, 'produtividade_mediana']
    })

extremos_temporais = pd.DataFrame(extremos_temporais)

extremos_temporais

Os extremos temporais mostram que as culturas apresentam diferentes padrões de evolução ao longo do período analisado. Em algumas culturas, como milho e trigo, os maiores valores de produtividade mediana ocorrem em 2016, enquanto outras atingem seus máximos em anos anteriores. Isso reforça que a evolução da produtividade não segue um comportamento uniforme entre as culturas.


## 5. Associação entre temperatura e produtividade

A associação entre temperatura média e produtividade é analisada por cultura. Primeiro, é calculada a correlação considerando os registros disponíveis de cada cultura. Em seguida, a análise será detalhada por país para verificar se diferenças estruturais entre os países influenciam o resultado agregado.

> **Observação:** correlação indica associação linear entre as variáveis e não deve ser interpretada como evidência de causalidade.

In [ ]:
temperatura_produtividade = (
    yield_integrado[
        ['Area', 'Item', 'Year', 'Value', 'avg_temp']
    ]
    .dropna(subset=['avg_temp'])
    .copy()
)

temperatura_produtividade.shape

In [ ]:
correlacao_temperatura = (
    temperatura_produtividade
    .groupby('Item')
    .apply(
        lambda grupo: grupo['avg_temp'].corr(grupo['Value']),
        include_groups=False
    )
    .sort_values()
)

correlacao_temperatura

### 5.1 Associação por país

Para verificar se o resultado agregado é influenciado por diferenças estruturais entre os países, a correlação é calculada individualmente para cada combinação de país e cultura.

São considerados apenas grupos com pelo menos 10 observações.

In [ ]:
correlacao_temperatura_pais = (
    temperatura_produtividade
    .groupby(['Area', 'Item'])
    .filter(lambda grupo: len(grupo) >= 10)
    .groupby('Item')
    .apply(
        lambda grupo: grupo.groupby('Area')
        .apply(
            lambda pais: pais['avg_temp'].corr(pais['Value']),
            include_groups=False
        )
        .dropna()
        .agg(['count', 'mean', 'median'])
    )
)

correlacao_temperatura_pais

A associação entre temperatura média e produtividade apresentou resultados distintos conforme o nível de análise. Na análise agregada, a maioria das culturas apresentou correlação negativa moderada. Entretanto, ao calcular as correlações individualmente dentro dos países com pelo menos 10 observações, as correlações médias e medianas foram positivas para todas as culturas.

Esse contraste indica que diferenças estruturais entre os países influenciam fortemente a associação observada nos dados agregados. Dessa forma, os resultados reforçam que as correlações identificadas representam associações lineares e não devem ser interpretadas como evidência de causalidade.


## 6. Associação entre precipitação e produtividade

A associação entre precipitação média e produtividade é analisada por cultura. Assim como na análise de temperatura, a correlação é calculada inicialmente de forma agregada e, posteriormente, avaliada por país.

> **Observação:** correlação indica associação linear entre as variáveis e não deve ser interpretada como evidência de causalidade.

In [ ]:
precipitacao_produtividade = (
    yield_integrado[
        ['Area', 'Item', 'Year', 'Value', 'rainfall_mm']
    ]
    .dropna(subset=['rainfall_mm'])
    .copy()
)

precipitacao_produtividade.shape

In [ ]:
correlacao_precipitacao = (
    precipitacao_produtividade
    .groupby('Item')
    .apply(
        lambda grupo: grupo['rainfall_mm'].corr(grupo['Value']),
        include_groups=False
    )
    .sort_values()
)

correlacao_precipitacao

### 6.1 Cobertura temporal da precipitação

In [ ]:
cobertura_chuva_pais = (
    precipitacao_produtividade
    .groupby('Area')['Year']
    .nunique()
    .describe()
)

cobertura_chuva_pais

### 6.2 Variação dos valores de precipitação

In [ ]:
variacao_chuva_pais = (
    precipitacao_produtividade
    .groupby(['Area', 'Item'])['rainfall_mm']
    .nunique()
    .reset_index(name='valores_chuva_distintos')
)

variacao_chuva_pais['valores_chuva_distintos'].describe()

A precipitação apresentou associações lineares predominantemente fracas com a produtividade das culturas analisadas. As correlações agregadas variaram de -0,274 a +0,092, sem uma relação consistente entre as culturas.

A análise da estrutura da variável revelou uma limitação importante: embora existam dados para vários anos e países, os valores de precipitação apresentam pouca variação temporal. Entre 983 combinações de país e cultura, a quantidade mediana de valores distintos de precipitação foi igual a 1, com máximo de 2 valores distintos.

Dessa forma, a variável de precipitação representa principalmente uma característica climática dos locais analisados, e não uma série temporal anual de precipitação. Por esse motivo, as correlações observadas devem ser interpretadas com cautela e não como evidência de uma relação causal ou temporal entre precipitação e produtividade.

## 7. Associação entre uso de pesticidas e produtividade

A associação entre o uso de pesticidas e a produtividade é analisada por cultura, considerando apenas os registros em que há informação disponível.

> **Observação:** correlação indica associação linear entre as variáveis e não deve ser interpretada como evidência de causalidade.

In [ ]:
pesticidas_produtividade = (
    yield_integrado[
        ['Area', 'Item', 'Year', 'Value', 'pesticides_tonnes']
    ]
    .dropna(subset=['pesticides_tonnes'])
    .copy()
)

pesticidas_produtividade.shape

In [ ]:
correlacao_pesticidas = (
    pesticidas_produtividade
    .groupby('Item')
    .apply(
        lambda grupo: grupo['pesticides_tonnes'].corr(grupo['Value']),
        include_groups=False
    )
    .sort_values()
)

correlacao_pesticidas

### 7.1 Cobertura temporal dos pesticidas

A quantidade de anos disponíveis é analisada por país para avaliar a cobertura temporal da variável de uso de pesticidas.

In [ ]:
cobertura_pesticidas_pais = (
    pesticidas_produtividade
    .groupby('Area')['Year']
    .nunique()
    .describe()

)

cobertura_pesticidas_pais

O uso de pesticidas apresentou associação linear positiva em todas as culturas analisadas, com correlações entre +0,061 e +0,244. A maior associação foi observada para arroz (Rice, paddy), enquanto a menor ocorreu em Plantains and others.

A cobertura temporal também se mostrou relativamente consistente: foram identificados dados para 166 países, com mediana de 27 anos disponíveis por país.

Apesar da associação positiva observada, os valores de correlação permanecem baixos. Portanto, os resultados indicam associação entre as variáveis, mas não permitem afirmar que maior uso de pesticidas cause maior produtividade, uma vez que outros fatores agrícolas, ambientais e socioeconômicos podem influenciar ambas.


## 8. Valores extremos e possíveis anomalias

Esta etapa busca identificar valores muito baixos ou elevados de produtividade que possam representar padrões específicos da base, registros atípicos ou possíveis inconsistências.

A identificação de valores extremos não implica que os registros sejam erros. Os casos encontrados serão avaliados considerando sua combinação de país, cultura e ano.

In [ ]:
anomalias_produtividade = (
    yield_integrado[
        ['Area', 'Item', 'Year', 'Value']
    ]
    .sort_values('Value', ascending=False)
)

anomalias_produtividade.head(15)

In [ ]:
valores_extremos = {
    'zeros': (yield_integrado['Value'] == 0).sum(),
    'abaixo_1000': (yield_integrado['Value'] < 1000).sum(),
    'abaixo_5000': (yield_integrado['Value'] < 5000).sum(),
    'acima_500000': (yield_integrado['Value'] > 500000).sum(),
}

valores_extremos

### 8.1 Registros com produtividade igual a zero

Os registros com produtividade igual a zero são analisados individualmente para verificar sua concentração por país, cultura e ano.

In [ ]:
zeros_produtividade = (
    yield_integrado[
        yield_integrado['Value'] == 0
    ][['Area', 'Item', 'Year', 'Value']]
)

zeros_produtividade

In [ ]:
baixos_por_cultura = (
    yield_integrado
    .groupby('Item')['Value']
    .agg(
        minimo='min',
        abaixo_1000=lambda x: (x < 1000).sum(),
        abaixo_5000=lambda x: (x < 5000).sum()
    )
    .sort_values('abaixo_1000', ascending=False)
)

baixos_por_cultura

Foram identificados 8 registros com produtividade superior a 500 mil e 8 registros com valor igual a zero. Também foram encontrados 40 registros abaixo de 1.000 e 1.348 registros abaixo de 5.000.

Os valores extremos elevados concentram-se principalmente em determinadas combinações de cultura e país, especialmente Plantains and others e Potatoes. O maior valor observado foi de 1.000.000 para Plantains and others no Quênia em 1964.

Os registros com produtividade igual a zero estão concentrados principalmente em New Caledonia, para Sorghum e Wheat, além de um registro de Sorghum no Occupied Palestinian Territory. Os valores muito baixos também apresentam maior concentração em algumas culturas específicas.

Como não foram identificadas evidências suficientes de inconsistências estruturais que justificassem a remoção desses registros, os valores extremos foram mantidos na base. Eles devem ser considerados na interpretação das estatísticas e visualizações, especialmente quando a média for utilizada.

> **Nota sobre os outliers:** a identificação estatística de um outlier pelo método do IQR não significa necessariamente que o registro seja um erro. Neste projeto, os valores extremos foram mantidos porque não foram encontradas evidências suficientes para classificá-los como inconsistências estruturais.